In [1]:
import tensorflow as tf
from skimage.io import imread
from keras.utils import generic_utils
from keras import layers
from keras import models
from tensorflow.keras import optimizers
from keras.optimizers import Adam
from tensorflow.keras.layers import Concatenate, Dense, Input, Conv2D, BatchNormalization
from vit_keras import vit
#load Check point
from tensorflow.keras.models import load_model

2025-01-21 02:03:36.873816: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-01-21 02:03:36.971810: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-21 02:03:37.485751: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/home/kannika/miniconda3/envs/AI/lib/:/home/kannika/miniconda3/envs/AI/lib/
2025-01-21

In [2]:
import os
## set gpu
gpu = 0
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu}" 
physical_devices = tf.config.list_physical_devices('GPU') 
print("Num GPUs:", len(physical_devices))

Num GPUs: 1


In [5]:
IMAGE_SIZE = 384
Numclass = 15

vit_model = vit.vit_l32(image_size=IMAGE_SIZE, classes=Numclass, activation = 'softmax', pretrained = True, 
                        include_top = False, pretrained_top = False)
print('[INFO]: Loading pre-trained weights')
x = vit_model.get_layer('ExtractToken').output
### add the tail layer ###  
Flatten_layer1 = layers.Flatten()(x)
BatchNormalization_layer1 = layers.BatchNormalization(name='BatchNormalization_1')(Flatten_layer1)
Dense_layer1 = layers.Dense(256, activation='gelu',name='Dense_View')(BatchNormalization_layer1)
BatchNormalization_layer2 = layers.BatchNormalization(name='BatchNormalization_2')(Dense_layer1)
Dense_layer2 = layers.Dense(Numclass, activation='softmax',name='Pred_15AB')(BatchNormalization_layer2)

model = models.Model(inputs= vit_model.input, outputs=[Dense_layer2], name = 'ViT_USAI15AB') 
model.summary()

2025-01-20 17:02:38.448803: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-01-20 17:02:39.260466: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9387 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:17:00.0, compute capability: 7.5


[INFO]: Loading pre-trained weights
Model: "ViT_USAI15AB"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 384, 384, 3)]     0         
                                                                 
 embedding (Conv2D)          (None, 12, 12, 1024)      3146752   
                                                                 
 reshape (Reshape)           (None, 144, 1024)         0         
                                                                 
 class_token (ClassToken)    (None, 145, 1024)         1024      
                                                                 
 Transformer/posembed_input   (None, 145, 1024)        148480    
 (AddPositionEmbs)                                               
                                                                 
 Transformer/encoderblock_0   ((None, 145, 1024),      12596224  
 (TransformerBlock

In [6]:
print('This is the number of trainable layers '
      'before freezing the conv base:', len(model.trainable_weights))

This is the number of trainable layers before freezing the conv base: 398


In [7]:
fc_layer = model.get_layer("Pred_15AB")
fc_layer.trainable = True
print('This is the number of trainable layers: ', len(model.trainable_weights))

This is the number of trainable layers:  398


In [11]:
for layer in model.layers:
    print(f"Layer Name: {layer.name} >> {layer.trainable}")
    print("="*125)

Layer Name: input_1 >> True
Layer Name: embedding >> True
Layer Name: reshape >> True
Layer Name: class_token >> True
Layer Name: Transformer/posembed_input >> True
Layer Name: Transformer/encoderblock_0 >> True
Layer Name: Transformer/encoderblock_1 >> True
Layer Name: Transformer/encoderblock_2 >> True
Layer Name: Transformer/encoderblock_3 >> True
Layer Name: Transformer/encoderblock_4 >> True
Layer Name: Transformer/encoderblock_5 >> True
Layer Name: Transformer/encoderblock_6 >> True
Layer Name: Transformer/encoderblock_7 >> True
Layer Name: Transformer/encoderblock_8 >> True
Layer Name: Transformer/encoderblock_9 >> True
Layer Name: Transformer/encoderblock_10 >> True
Layer Name: Transformer/encoderblock_11 >> True
Layer Name: Transformer/encoderblock_12 >> True
Layer Name: Transformer/encoderblock_13 >> True
Layer Name: Transformer/encoderblock_14 >> True
Layer Name: Transformer/encoderblock_15 >> True
Layer Name: Transformer/encoderblock_16 >> True
Layer Name: Transformer/encod

In [12]:
model.input_shape

(None, 384, 384, 3)

In [13]:
height = width = model.input_shape[1]
input_shape = (height, width, 3)
print(input_shape)

(384, 384, 3)


## Load Model

In [4]:
def loadresumemodel_1input(model_dir):
    model = load_model(model_dir)
    height = width = model.input_shape[1]  ## model.input_shape[0][1]
    input_shape = (height, width, 3)
    
    return input_shape, model

In [5]:
model_dir = "/media/tohn/HDD2/Model_unlearn/Models_USAI/ViTModel/ViTorigin_USAI/R1_balanced/exp1/models/modelVITL32_Unlearning_USAI15ab_balanced-R1_balanced.h5"
input_shape, model = loadresumemodel_1input(model_dir)

2025-01-21 02:04:44.009337: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-01-21 02:04:44.417366: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9388 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:17:00.0, compute capability: 7.5


In [6]:
input_shape

(384, 384, 3)

In [7]:
model.summary()

Model: "ViT_USAI"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 384, 384, 3)]     0         
                                                                 
 embedding (Conv2D)          (None, 12, 12, 1024)      3146752   
                                                                 
 reshape (Reshape)           (None, 144, 1024)         0         
                                                                 
 class_token (ClassToken)    (None, 145, 1024)         1024      
                                                                 
 Transformer/posembed_input   (None, 145, 1024)        148480    
 (AddPositionEmbs)                                               
                                                                 
 Transformer/encoderblock_0   ((None, 145, 1024),      12596224  
 (TransformerBlock)           (None, 16, None, None))     